In [1]:
#Raw orders (messy data)
raw_orders = [
    {"order_id": 1001, "customer": "Ali",    "category": "electronics",  "price": "500",     "qty": 2},
    {"order_id": 1002, "customer": "Sara",   "category": " Clothing ",   "price": 40.0,      "qty": 3},
    {"order_id": 1003, "customer": "Bilal",  "category": "books",        "price": "$12.50",  "qty": 2},   # threshold se kam
    {"order_id": 1004, "customer": "Hina",   "category": "Grocery",      "price": 30,        "qty": 4},
    {"order_id": 1005, "customer": "Usman",  "category": "electronics",  "price": None,      "qty": 1},   # price missing
    {"order_id": 1006, "customer": "Zara",   "category": "BOOKS",        "price": 20,        "qty": 5},
    {"order_id": 1007, "customer": "Omar",   "category": "toys",         "price": 60,        "qty": 1},   # rule nahi, 0% discount
    {"order_id": 1008, "customer": "Nida",   "category": "clothing",     "price": 15,        "qty": 2},   # threshold se kam
    {"order_id": 1009, "customer": "Ali",    "category": "Electronics",  "price": "$1,200",  "qty": 1},
    {"order_id": 1010, "customer": "Kiran",  "category": "grocery",      "price": 10,        "qty": 0},   # qty zero
]

In [2]:
#Config + pure functions
import re
from functools import reduce

DISCOUNT_RULES = {"electronics": 0.10, "clothing": 0.20, "books": 0.05, "grocery": 0.0}
DEFAULT_DISCOUNT = 0.0
MIN_ORDER_AMOUNT = 50.0


def parse_price(value):
    """'$1,200' / '25.5' / 40 -> float, warna None."""
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return float(value)
    match = re.search(r"\d[\d,]*\.?\d*", str(value or ""))
    return float(match.group().replace(",", "")) if match else None


def parse_qty(value):
    try:
        return int(float(value))
    except (TypeError, ValueError):
        return None


def clean_order(order):
    """Valid order ka saaf dict, warna None."""
    price = parse_price(order.get("price"))
    qty = parse_qty(order.get("qty"))
    category = " ".join(str(order.get("category") or "").split()).lower()
    if price is None or qty is None or qty <= 0 or not category:
        return None
    return {
        "order_id": order["order_id"],
        "customer": str(order.get("customer") or "Unknown").strip().title(),
        "category": category,
        "price": price,
        "qty": qty,
    }


def apply_discount(order):
    """Category rule ke mutabiq discount lagao."""
    rate = DISCOUNT_RULES.get(order["category"], DEFAULT_DISCOUNT)
    subtotal = round(order["price"] * order["qty"], 2)
    discount = round(subtotal * rate, 2)
    return {**order, "subtotal": subtotal, "rate": rate,
            "discount": discount, "final": round(subtotal - discount, 2)}


def add_to_category_summary(acc, order):
    """reduce ka step: ek order ko category summary mein jodo."""
    prev = acc.get(order["category"], {"orders": 0, "revenue": 0.0, "discount": 0.0})
    updated = {
        "orders": prev["orders"] + 1,
        "revenue": round(prev["revenue"] + order["final"], 2),
        "discount": round(prev["discount"] + order["discount"], 2),
    }
    return {**acc, order["category"]: updated}

In [3]:
#Pipeline
def run_pipeline(orders):
    cleaned = list(filter(None, map(clean_order, orders)))                # invalid drop
    discounted = list(map(apply_discount, cleaned))                       # discount lagao
    qualified = list(filter(lambda o: o["final"] >= MIN_ORDER_AMOUNT, discounted))
    rejected = list(filter(lambda o: o["final"] < MIN_ORDER_AMOUNT, discounted))

    return {
        "raw_count": len(orders),
        "invalid_count": len(orders) - len(cleaned),
        "qualified": qualified,
        "rejected": rejected,
        "total_revenue": round(sum(map(lambda o: o["final"], qualified)), 2),
        "total_discount": round(sum(map(lambda o: o["discount"], qualified)), 2),
        "by_category": reduce(add_to_category_summary, qualified, {}),
    }


result = run_pipeline(raw_orders)
print(f"Qualified: {len(result['qualified'])} | Rejected: {len(result['rejected'])} "
      f"| Invalid: {result['invalid_count']}")
print(f"Total revenue: Rs. {result['total_revenue']:,.2f}")

Qualified: 6 | Rejected: 2 | Invalid: 2
Total revenue: Rs. 2,351.00


In [4]:
#Neat summary
LINE = "=" * 62
print(LINE)
print("ORDER DISCOUNT PIPELINE - SUMMARY".center(62))
print(LINE)

print("\nQUALIFIED ORDERS")
print("-" * 62)
print("\n".join(map(
    lambda o: f"  #{o['order_id']:<5} {o['customer']:<8} {o['category'].title():<12}"
              f"{o['subtotal']:>9,.2f} -{o['discount']:>7,.2f} = {o['final']:>9,.2f}",
    sorted(result["qualified"], key=lambda o: o["final"], reverse=True),
)))

print("\nREJECTED (Rs. {:.0f} se kam)".format(MIN_ORDER_AMOUNT))
print("-" * 62)
print("\n".join(map(
    lambda o: f"  #{o['order_id']:<5} {o['customer']:<8} final = {o['final']:,.2f}",
    result["rejected"],
)))

print("\nBY CATEGORY")
print("-" * 62)
print("\n".join(map(
    lambda kv: f"  {kv[0].title():<12} orders: {kv[1]['orders']}  "
               f"revenue: {kv[1]['revenue']:>10,.2f}  discount: {kv[1]['discount']:>8,.2f}",
    sorted(result["by_category"].items()),
)))

print("\n" + LINE)
print(f"Total revenue  : Rs. {result['total_revenue']:,.2f}")
print(f"Total discount : Rs. {result['total_discount']:,.2f}")
print(LINE)

              ORDER DISCOUNT PIPELINE - SUMMARY               

QUALIFIED ORDERS
--------------------------------------------------------------
  #1009  Ali      Electronics  1,200.00 - 120.00 =  1,080.00
  #1001  Ali      Electronics  1,000.00 - 100.00 =    900.00
  #1004  Hina     Grocery        120.00 -   0.00 =    120.00
  #1002  Sara     Clothing       120.00 -  24.00 =     96.00
  #1006  Zara     Books          100.00 -   5.00 =     95.00
  #1007  Omar     Toys            60.00 -   0.00 =     60.00

REJECTED (Rs. 50 se kam)
--------------------------------------------------------------
  #1003  Bilal    final = 23.75
  #1008  Nida     final = 24.00

BY CATEGORY
--------------------------------------------------------------
  Books        orders: 1  revenue:      95.00  discount:     5.00
  Clothing     orders: 1  revenue:      96.00  discount:    24.00
  Electronics  orders: 2  revenue:   1,980.00  discount:   220.00
  Grocery      orders: 1  revenue:     120.00  discount:     0.